# TODO 6.3：GPTQ INT4 实验

使用与 AWQ 相同的校准集、bit 数和 group size 32/64/128。完整模型写入 `banking_llm_project/gptq`，分析文件写入 `OptimizationSystem/out/gptq`。

首次运行先执行安装单元，然后通过 `运行时 → 重启会话` 重启。重启后跳过安装单元，从挂载 Drive 单元继续。

In [ ]:
%pip -q install -U gptqmodel==7.5.0 optimum accelerate safetensors
%pip -q install -U transformers==5.18.0
print('安装完成。现在必须重启 Colab 会话，不要继续运行下一个单元。')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import importlib.metadata
import json
import sys
from pathlib import Path
import torch

print('transformers:', importlib.metadata.version('transformers'))
print('gptqmodel:', importlib.metadata.version('gptqmodel'))
from transformers import AutoModelForCausalLM, AutoTokenizer, GPTQConfig
print('Transformers GPTQ 依赖导入：PASS')

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
CONFIG_PATH = PROJECT_DIR / 'configs/gptq.json'
CALIBRATION_MANIFEST = PROJECT_DIR / 'out/dataset_input/dataset_manifest.json'
RESULT_DIR = PROJECT_DIR / 'out/gptq'
MODEL_RESULT_DIR = Path('/content/drive/MyDrive/banking_llm_project/gptq')
sys.path.insert(0, str(PROJECT_DIR))

from src.data.finqa_assets import load_assets
from src.quantization.gptq import (
    build_gptq_plans, collect_gptq_evidence,
    run_gptq_plan, validate_gptq_evidence,
)

assets = load_assets(DATASET_DIR)
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
plans = build_gptq_plans(config)
assert torch.cuda.is_available(), '请切换到 NVIDIA GPU。'
print(torch.cuda.get_device_name(0), [plan.plan_id for plan in plans])

In [ ]:
import gc

results = []
for plan in plans:
    manifest = RESULT_DIR / plan.plan_id / 'quantized_model_manifest.json'
    if manifest.is_file():
        print('跳过已完成:', plan.plan_id)
        continue
    result = run_gptq_plan(
        plan=plan,
        model_manifest_file=MODEL_MANIFEST,
        calibration_prompts=assets.calibration_prompts,
        output_dir=RESULT_DIR / plan.plan_id,
        model_output_dir=MODEL_RESULT_DIR / plan.plan_id,
        max_calibration_samples=int(config['max_calibration_samples']),
        max_calibration_seq_len=int(config['max_calibration_seq_len']),
    )
    results.append(result)
    if result['status'] != 'pass':
        raise RuntimeError(result['detail'])
    gc.collect()
    torch.cuda.empty_cache()
print(json.dumps(results, ensure_ascii=False, indent=2))

In [ ]:
evidence = collect_gptq_evidence(
    CONFIG_PATH, CALIBRATION_MANIFEST, MODEL_MANIFEST, RESULT_DIR
)
evidence_path = RESULT_DIR / 'gptq_evidence.json'
evidence_path.write_text(
    json.dumps(evidence, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
report = validate_gptq_evidence(evidence_path)
(RESULT_DIR / 'validation.json').write_text(
    json.dumps(report.to_dict(), ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print(json.dumps(report.to_dict(), ensure_ascii=False, indent=2))
assert report.complete, 'GPTQ 三个 group size 未全部完成。'
print('TODO 6.3 GPTQ 导出验收：PASS')